# 🚀 File 3: Training Transformers Pipeline (Multi-Seed DistilBERT & RoBERTa)
**Riset Skripsi**: Perbandingan Kinerja RoBERTa-base dan DistilBERT-base untuk Deteksi Phishing URL  
- **Target Runtime**: Google Colab (GPU NVIDIA Tesla T4 16GB / CUDA 12.x)  
- **Storage Backend**: Google Drive (`/content/drive/MyDrive/Skripsi/`)  
- **Tujuan Utama**: Melatih arsitektur Transformer secara **1-Stage End-to-End Fine-Tuning** dengan **Mixed Precision (FP16 / PyTorch AMP)**:
  1. **DistilBERT-base** dengan **Custom URLBERT Tokenizer** (WordPiece > 30.000 vocab) diuji pada **3 Seed berbeda** (`42`, `123`, `999`) untuk mengukur reliabilitas statistik (Mean ± Std Dev).
  2. **RoBERTa-base** (Byte-Pair Encoding) dilatih dengan konfigurasi 1-stage identik untuk *fair comparison*.

---
### 📋 Ringkasan Alur Kerja Notebook:
1. **Setup Lingkungan & Mount Google Drive**: Menghubungkan Google Drive dan memverifikasi GPU Tesla T4.
2. **Pemuatan Dataset**: Membaca `train.csv`, `val.csv`, dan `test.csv` dari `/content/drive/MyDrive/Skripsi/new data/`.
3. **Pemuatan Custom URLBERT Tokenizer**:
   - Memuat `vocab_merged.txt` dari Google Drive menggunakan `tokenizers.models.WordPiece`.
   - Memvalidasi `assert len(tokenizer) > 30000`.
4. **PyTorch Dataset & DataLoader**:
   - Kelas `URLDataset` dengan padding dinamis dan tensor CUDA.
   - Batch size 64 teroptimasi GPU memory.
5. **Engine Pelatihan 1-Stage End-to-End dengan PyTorch AMP (FP16)**:
   - `torch.cuda.amp.autocast()` dan `torch.cuda.amp.GradScaler()`.
   - Hyperparameter riset: Epochs=2, Batch Size=64, LR=2e-5, Warmup Ratio=10%, Weight Decay=0.01.
   - Tracking waktu pelatihan (*wall-clock time*) dan konsumsi Peak VRAM GPU.
6. **Pelatihan Multi-Seed DistilBERT**:
   - Training loop untuk seed `[42, 123, 999]`.
   - Menyimpan seluruh checkpoint model & tokenizer ke:
     * `/content/drive/MyDrive/Skripsi/models/distilbert_seed42/`
     * `/content/drive/MyDrive/Skripsi/models/distilbert_seed123/`
     * `/content/drive/MyDrive/Skripsi/models/distilbert_seed999/`
   - Agregasi statistik Mean ± Std Dev.
7. **Pelatihan RoBERTa-base (Seed 42)**:
   - Pelatihan 1-stage FP16 model `roberta-base`.
   - Menyimpan model & tokenizer ke `/content/drive/MyDrive/Skripsi/models/roberta_phishing/`.
8. **Ekspor Hasil**: Menyimpan tabel rekapitulasi ke `/content/drive/MyDrive/Skripsi/results/transformer_training_results.csv`.


## 1. Setup Lingkungan & Instalasi Dependensi
Menginstalasi pustaka transformer dan deep learning: `transformers`, `tokenizers`, `torch`, `accelerate`, `scikit-learn`, `pandas`, `numpy`, `tqdm`.


In [1]:
# Instalasi pustaka transformer di Google Colab
!pip install transformers tokenizers torch accelerate scikit-learn pandas numpy tqdm -q

import os
import sys
import gc
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from transformers import (
    PreTrainedTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from google.colab import drive

# Verifikasi Hardware GPU
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = torch.device("cuda:0")
else:
    print("⚠️ GPU tidak terdeteksi, berjalan pada CPU.")
    device = torch.device("cpu")

print("\n[OK] Setup library berhasil.")


PyTorch Version : 2.11.0+cu130
CUDA Available  : True
Device Name     : NVIDIA L4
VRAM Capacity   : 22.03 GB

[OK] Setup library berhasil.


## 2. Mount Google Drive & Konfigurasi Path Proyek
Menghubungkan direktori Google Drive (`/content/drive/MyDrive/Skripsi`) dan menyiapkan seluruh jalur direktori model, tokenizer, dan hasil.


In [2]:
# 1. Mount Google Drive
print("[INFO] Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Direktori Proyek
BASE_DIR = "/content/drive/MyDrive/Skripsi"
DATA_DIR = os.path.join(BASE_DIR, "new data")
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(BASE_DIR, "data")

CUSTOM_TOKENIZER_DIR = os.path.join(BASE_DIR, "custom_url_tokenizer")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 3. Path File Data Split
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV = os.path.join(DATA_DIR, "val.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

# 4. Path Kosakata Tokenizer Kustom
VOCAB_MERGED = os.path.join(CUSTOM_TOKENIZER_DIR, "vocab_merged.txt")
VOCAB_GIT = os.path.join(CUSTOM_TOKENIZER_DIR, "vocab_Git.txt")
VOCAB_LOCAL = os.path.join(CUSTOM_TOKENIZER_DIR, "vocab.txt")

# 5. Path Output Rekapitulasi
TRANSFORMER_RESULTS_CSV = os.path.join(RESULTS_DIR, "transformer_training_results.csv")

print(f"[OK] Direktori Utama    : {BASE_DIR}")
print(f"[OK] Direktori Data     : {DATA_DIR}")
print(f"[OK] Direktori Models   : {MODELS_DIR}")
print(f"[OK] Direktori Results  : {RESULTS_DIR}")


Mounted at /content/drive
[OK] Direktori Utama    : /content/drive/MyDrive/Skripsi
[OK] Direktori Data     : /content/drive/MyDrive/Skripsi/new data
[OK] Direktori Models   : /content/drive/MyDrive/Skripsi/models
[OK] Direktori Results  : /content/drive/MyDrive/Skripsi/results


## 3. Pemuatan Dataset & Delimiter-Spacing Preprocessing
Membaca `train.csv`, `val.csv`, dan `test.csv`. URL dipersiapkan dengan delimiter spacing (`/`, `-`, `?`, `=`, `&`, `.`, `://`) sesuai rancangan vocabulary WordPiece URLBERT agar pemecahan subword berlangsung optimal.


In [3]:
print("[INFO] Memuat dataset Train, Val, dan Test...")

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"[OK] Train Set : {len(train_df):,} sampel")
print(f"[OK] Val Set   : {len(val_df):,} sampel")
print(f"[OK] Test Set  : {len(test_df):,} sampel")

import re

def preprocess_delimiter_spacing(url_str: str) -> str:
    '''
    Menambahkan spasi di sekitar delimiter URL agar WordPiece tokenizer
    dapat membedakan token subword spesifik domain, path, dan parameter.
    '''
    if not isinstance(url_str, str):
        return ""
    u = url_str.strip()
    u = u.replace('://', ' :// ')
    u = u.replace('/', ' / ')
    u = u.replace('?', ' ? ')
    u = u.replace('=', ' = ')
    u = u.replace('&', ' & ')
    u = u.replace('.', ' . ')
    u = u.replace('-', ' - ')
    return re.sub(r'\s+', ' ', u).strip()

text_col = 'url_norm' if 'url_norm' in train_df.columns else 'url'
print(f"[INFO] Menggunakan kolom: '{text_col}' untuk preprocessing URL...")

train_df['url_spaced'] = train_df[text_col].apply(preprocess_delimiter_spacing)
val_df['url_spaced'] = val_df[text_col].apply(preprocess_delimiter_spacing)
test_df['url_spaced'] = test_df[text_col].apply(preprocess_delimiter_spacing)

print("\nContoh URL sebelum vs sesudah delimiter-spacing:")
for i in range(2):
    print(f" [Asli]   : {train_df[text_col].iloc[i]}")
    print(f" [Spaced] : {train_df['url_spaced'].iloc[i]}\n")


[INFO] Memuat dataset Train, Val, dan Test...
[OK] Train Set : 927,898 sampel
[OK] Val Set   : 198,052 sampel
[OK] Test Set  : 222,316 sampel
[INFO] Menggunakan kolom: 'url_norm' untuk preprocessing URL...

Contoh URL sebelum vs sesudah delimiter-spacing:
 [Asli]   : bs-369.com/views/account/login.aspx?returnurl=
 [Spaced] : bs - 369 . com / views / account / login . aspx ? returnurl =

 [Asli]   : amelihair.ru/images/6550054213/745353769679
 [Spaced] : amelihair . ru / images / 6550054213 / 745353769679



## 4. Inisialisasi Tokenizer Kustom (WordPiece Engine > 30.000 Vocab)
Membangun tokenizer kustom dari `vocab_merged.txt` menggunakan modul low-level `tokenizers.models.WordPiece` dan membungkusnya ke dalam `PreTrainedTokenizerFast` HuggingFace:
- Menetapkan `Whitespace()` pre-tokenizer.
- Menetapkan `TemplateProcessing` untuk token spesial `[CLS]` dan `[SEP]`.
- Melakukan asersi validitas ukuran kosakata: `assert len(tokenizer) > 30000`.


In [5]:
print("[INFO] Membangun Custom URLBERT Tokenizer dari Google Drive...")

# Tentukan file kosakata
vocab_file = None
for vf in [VOCAB_MERGED, VOCAB_GIT, VOCAB_LOCAL]:
    if os.path.exists(vf):
        vocab_file = vf
        break

if vocab_file is None:
    raise FileNotFoundError(
        f"❌ File kosakata tidak ditemukan di {CUSTOM_TOKENIZER_DIR}!\n"
        f"Pastikan vocab_merged.txt berada di: {VOCAB_MERGED}"
    )

print(f"[OK] Memuat file kosakata: {vocab_file}")

# 1. Bangun WordPiece model dari file vocab
wp_model = WordPiece.from_file(vocab_file, unk_token="[UNK]")
raw_tokenizer = Tokenizer(wp_model)

# 2. Pre-tokenizer menggunakan Whitespace
raw_tokenizer.pre_tokenizer = Whitespace()

# 3. Post-processor [CLS] dan [SEP]
cls_id = raw_tokenizer.token_to_id("[CLS]")
sep_id = raw_tokenizer.token_to_id("[SEP]")

if cls_id is not None and sep_id is not None:
    raw_tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", cls_id),
            ("[SEP]", sep_id)
        ]
    )

# 4. Bungkus dengan PreTrainedTokenizerFast Hugging Face
custom_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    unk_token="[UNK]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    mask_token="[MASK]",
    do_lower_case=True
)

# 5. Asersi Validasi Ukuran Kosakata
print(f"[INFO] Jumlah kosakata tokenizer: {len(custom_tokenizer):,}")
assert len(custom_tokenizer) > 30000, f"❌ Vocab size terlalu kecil ({len(custom_tokenizer)})! Harap periksa file vocab."
print("✅ [PASS] Asersi Berhasil: len(custom_tokenizer) > 30,000 (Vocab collapse dicegah).")

# Tokenizer untuk RoBERTa (Byte-Pair Encoding standar)
print("\n[INFO] Memuat tokenizer Byte-Pair Encoding untuk RoBERTa-base...")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"[OK] RoBERTa Tokenizer siap: {len(roberta_tokenizer):,} vocab.")


[INFO] Membangun Custom URLBERT Tokenizer dari Google Drive...
[OK] Memuat file kosakata: /content/drive/MyDrive/Skripsi/custom_url_tokenizer/vocab_merged.txt
[INFO] Jumlah kosakata tokenizer: 30,372
✅ [PASS] Asersi Berhasil: len(custom_tokenizer) > 30,000 (Vocab collapse dicegah).

[INFO] Memuat tokenizer Byte-Pair Encoding untuk RoBERTa-base...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[OK] RoBERTa Tokenizer siap: 50,265 vocab.


## 5. Konstruksi PyTorch Dataset & DataLoader
Membangun kelas `URLDataset` teroptimasi PyTorch untuk memproses tokenisasi on-the-fly dengan panjang maksimum `max_length=128`.


In [6]:
class URLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_dataloaders(train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, tokenizer, batch_size=64):
    train_dataset = URLDataset(train_texts, train_labels, tokenizer)
    val_dataset = URLDataset(val_texts, val_labels, tokenizer)
    test_dataset = URLDataset(test_texts, test_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    return train_loader, val_loader, test_loader

print("[OK] Definisi URLDataset dan create_dataloaders siap.")


[OK] Definisi URLDataset dan create_dataloaders siap.


## 6. Engine Pelatihan 1-Stage End-to-End dengan PyTorch AMP (FP16)
Implementasi fungsi modular training:
- **Mixed Precision**: `torch.cuda.amp.autocast()` dan `torch.cuda.amp.GradScaler()` untuk efisiensi komputasi GPU T4.
- **Optimizer**: `AdamW(lr=2e-5, weight_decay=0.01)`.
- **Scheduler**: Linear Warmup Scheduler (10% total steps).
- **Early Evaluation & Checkpointing**: Menyimpan state terbaik berdasarkan Validation F1-Score.


In [7]:
def get_linear_warmup_scheduler(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(
            0.0,
            float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps))
        )
    return LambdaLR(optimizer, lr_lambda)

def evaluate_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    use_amp = torch.cuda.is_available()
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            total_loss += outputs.loss.item()
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / max(len(dataloader), 1)
    return avg_loss, np.array(all_labels), np.array(all_preds), np.array(all_probs)

def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1_bin = f1_score(y_true, y_pred, average='binary', zero_division=0)
    f1_mac = f1_score(y_true, y_pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_binary': f1_bin,
        'f1_macro': f1_mac,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }

def train_one_stage_model(
    model, train_loader, val_loader, test_loader, tokenizer,
    save_dir, model_name="DistilBERT",
    epochs=2, lr=2e-5, warmup_ratio=0.10, weight_decay=0.01,
    device=device
):
    print(f"\n======================================================================")
    print(f"🚀 [1-STAGE FINE-TUNING] {model_name} (End-to-End FP16/AMP)")
    print(f"   Epochs: {epochs} | LR: {lr} | Warmup: {warmup_ratio*100:.0f}% | WD: {weight_decay}")
    print(f"   Target Save Directory: {save_dir}")
    print(f"======================================================================")

    model.to(device)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler = get_linear_warmup_scheduler(optimizer, warmup_steps, total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val_f1 = -1.0
    t_start_train = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_train_loss += loss.item()

            if (step + 1) % 500 == 0 or (step + 1) == len(train_loader):
                cur_loss = total_train_loss / (step + 1)
                print(f"   [Epoch {epoch}/{epochs}] Step {step+1}/{len(train_loader)} | Train Loss: {cur_loss:.4f}")

        # Evaluasi Akhir Epoch pada Validation Set
        val_loss, val_true, val_pred, val_prob = evaluate_model(model, val_loader, device)
        val_metrics = compute_metrics(val_true, val_pred, val_prob)
        print(f"   >> [Val Epoch {epoch}] Loss: {val_loss:.4f} | Acc: {val_metrics['accuracy']*100:.2f}% | F1-Binary: {val_metrics['f1_binary']:.4f} | ROC-AUC: {val_metrics['roc_auc']:.4f}")

        # Simpan checkpoint jika F1 meningkat
        if val_metrics['f1_binary'] > best_val_f1:
            best_val_f1 = val_metrics['f1_binary']
            os.makedirs(save_dir, exist_ok=True)
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
            print(f"   ⭐ Checkpoint terbaik disimpan ke: {save_dir} (Val F1: {best_val_f1:.4f})")

    train_duration = time.perf_counter() - t_start_train
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
    print(f"\n[OK] Pelatihan selesai dalam {train_duration:.2f} dtk ({train_duration/60:.2f} menit) | Peak VRAM: {peak_vram_gb:.3f} GB")

    # Evaluasi Independen pada Test Set menggunakan checkpoint terbaik
    print(f"[INFO] Memuat checkpoint terbaik dari {save_dir} untuk evaluasi Test Set...")
    best_model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    best_model.to(device)

    # Benchmark Latensi Inferensi
    t0_inf = time.perf_counter()
    test_loss, test_true, test_pred, test_prob = evaluate_model(best_model, test_loader, device)
    total_inf_time = time.perf_counter() - t0_inf

    test_metrics = compute_metrics(test_true, test_pred, test_prob)
    latency_ms = (total_inf_time / len(test_true)) * 1000.0
    throughput = len(test_true) / total_inf_time

    test_metrics['test_loss'] = test_loss
    test_metrics['training_time_s'] = train_duration
    test_metrics['peak_vram_gb'] = peak_vram_gb
    test_metrics['latency_ms'] = latency_ms
    test_metrics['throughput_sps'] = throughput

    print(f"📊 HASIL TEST SET - {model_name}:")
    print(f"   Accuracy         : {test_metrics['accuracy']*100:.2f}%")
    print(f"   Precision        : {test_metrics['precision']:.4f}")
    print(f"   Recall           : {test_metrics['recall']:.4f}")
    print(f"   F1-Score (Binary): {test_metrics['f1_binary']:.4f}")
    print(f"   F1-Score (Macro) : {test_metrics['f1_macro']:.4f}")
    print(f"   ROC-AUC          : {test_metrics['roc_auc']:.4f}")
    print(f"   PR-AUC           : {test_metrics['pr_auc']:.4f}")
    print(f"   Latency          : {latency_ms:.4f} ms/sampel")
    print(f"   Throughput       : {throughput:,.1f} sampel/detik")

    del model, best_model, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return test_metrics

print("[OK] Fungsi training 1-stage AMP siap.")


[OK] Fungsi training 1-stage AMP siap.


## 7. Pelatihan Multi-Seed DistilBERT (Seeds: 42, 123, 999)
Melatih model `distilbert-base-uncased` dengan custom URL tokenizer pada 3 seed berbeda untuk verifikasi stabilitas dan reliabilitas performa.
Setiap checkpoint disimpan permanen ke Google Drive:
- `/content/drive/MyDrive/Skripsi/models/distilbert_seed42/`
- `/content/drive/MyDrive/Skripsi/models/distilbert_seed123/`
- `/content/drive/MyDrive/Skripsi/models/distilbert_seed999/`


In [8]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DISTILBERT_SEEDS = [42, 123, 999]
distilbert_results = []

# Persiapkan DataLoader untuk DistilBERT
train_loader_distil, val_loader_distil, test_loader_distil = create_dataloaders(
    train_df['url_spaced'], train_df['label'],
    val_df['url_spaced'], val_df['label'],
    test_df['url_spaced'], test_df['label'],
    custom_tokenizer,
    batch_size=64
)

for seed in DISTILBERT_SEEDS:
    print(f"\n======================================================================")
    print(f"🌱 Menjalankan Pelatihan DistilBERT dengan SEED = {seed}")
    print(f"======================================================================")
    set_seed(seed)

    # Inisialisasi arsitektur DistilBERT
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    # Sesuaikan embedding layer dengan ukuran kosakata kustom (> 30.000)
    model.resize_token_embeddings(len(custom_tokenizer))

    save_path = os.path.join(MODELS_DIR, f"distilbert_seed{seed}")

    metrics = train_one_stage_model(
        model=model,
        train_loader=train_loader_distil,
        val_loader=val_loader_distil,
        test_loader=test_loader_distil,
        tokenizer=custom_tokenizer,
        save_dir=save_path,
        model_name=f"DistilBERT (Seed {seed})",
        epochs=2,
        lr=2e-5,
        warmup_ratio=0.10,
        weight_decay=0.01
    )

    metrics['model'] = 'DistilBERT-base'
    metrics['seed'] = seed
    distilbert_results.append(metrics)

# Hitung Rata-rata & Standar Deviasi (Mean ± Std Dev)
df_distil = pd.DataFrame(distilbert_results)
metric_cols = ['accuracy', 'precision', 'recall', 'f1_binary', 'f1_macro', 'roc_auc', 'pr_auc', 'latency_ms', 'throughput_sps']

print("\n" + "="*70)
print("📈 REKAPITULASI MULTI-SEED DISTILBERT (MEAN ± STD DEV):")
print("="*70)
for m in metric_cols:
    vals = df_distil[m].values
    mean_v = np.mean(vals)
    std_v = np.std(vals)
    print(f" - {m:<15}: {mean_v:.4f} ± {std_v:.4f}")



🌱 Menjalankan Pelatihan DistilBERT dengan SEED = 42


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

/tmp/ipykernel_2318/1431587931.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


   >> [Val Epoch 2] Loss: 0.8213 | Acc: 87.01% | F1-Binary: 0.8817 | ROC-AUC: 0.9494

[OK] Pelatihan selesai dalam 3241.89 dtk (54.03 menit) | Peak VRAM: 2.466 GB
[INFO] Memuat checkpoint terbaik dari /content/drive/MyDrive/Skripsi/models/distilbert_seed999 untuk evaluasi Test Set...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_2318/1431587931.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


📊 HASIL TEST SET - DistilBERT (Seed 999):
   Accuracy         : 91.17%
   Precision        : 0.8501
   Recall           : 0.9805
   F1-Score (Binary): 0.9106
   F1-Score (Macro) : 0.9117
   ROC-AUC          : 0.9769
   PR-AUC           : 0.9584
   Latency          : 0.4218 ms/sampel
   Throughput       : 2,370.9 sampel/detik

📈 REKAPITULASI MULTI-SEED DISTILBERT (MEAN ± STD DEV):
 - accuracy       : 0.9183 ± 0.0050
 - precision      : 0.8624 ± 0.0096
 - recall         : 0.9782 ± 0.0024
 - f1_binary      : 0.9166 ± 0.0045
 - f1_macro       : 0.9183 ± 0.0050
 - roc_auc        : 0.9812 ± 0.0038
 - pr_auc         : 0.9681 ± 0.0088
 - latency_ms     : 0.4290 ± 0.0057
 - throughput_sps : 2331.5448 ± 30.9800


## 8. Pelatihan RoBERTa-base (Seed 42)
Melatih model `roberta-base` (BPE Tokenizer) dengan skema 1-stage FP16 yang sama untuk pembandingan langsung terhadap arsitektur DistilBERT.
Checkpoint model disimpan ke: `/content/drive/MyDrive/Skripsi/models/roberta_phishing/`.


In [9]:
set_seed(42)

print("\n======================================================================")
print("🚀 Memulai Pelatihan RoBERTa-base (Seed 42)...")
print("======================================================================")

# DataLoader untuk RoBERTa (menggunakan tokenizer Byte-Pair Encoding)
train_loader_rob, val_loader_rob, test_loader_rob = create_dataloaders(
    train_df[text_col], train_df['label'],
    val_df[text_col], val_df['label'],
    test_df[text_col], test_df['label'],
    roberta_tokenizer,
    batch_size=64
)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

roberta_save_path = os.path.join(MODELS_DIR, "roberta_phishing")

roberta_metrics = train_one_stage_model(
    model=roberta_model,
    train_loader=train_loader_rob,
    val_loader=val_loader_rob,
    test_loader=test_loader_rob,
    tokenizer=roberta_tokenizer,
    save_dir=roberta_save_path,
    model_name="RoBERTa-base (Seed 42)",
    epochs=2,
    lr=2e-5,
    warmup_ratio=0.10,
    weight_decay=0.01
)

roberta_metrics['model'] = 'RoBERTa-base'
roberta_metrics['seed'] = 42



🚀 Memulai Pelatihan RoBERTa-base (Seed 42)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/tmp/ipykernel_2318/1431587931.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


   >> [Val Epoch 2] Loss: 0.2218 | Acc: 96.43% | F1-Binary: 0.9650 | ROC-AUC: 0.9905

[OK] Pelatihan selesai dalam 6461.20 dtk (107.69 menit) | Peak VRAM: 5.251 GB
[INFO] Memuat checkpoint terbaik dari /content/drive/MyDrive/Skripsi/models/roberta_phishing untuk evaluasi Test Set...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_2318/1431587931.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


📊 HASIL TEST SET - RoBERTa-base (Seed 42):
   Accuracy         : 97.22%
   Precision        : 0.9569
   Recall           : 0.9837
   F1-Score (Binary): 0.9701
   F1-Score (Macro) : 0.9721
   ROC-AUC          : 0.9952
   PR-AUC           : 0.9916
   Latency          : 0.8115 ms/sampel
   Throughput       : 1,232.3 sampel/detik


## 9. Rekapitulasi & Ekspor Hasil ke Google Drive
Menyimpan ringkasan komparasi lengkap arsitektur Transformer ke `/content/drive/MyDrive/Skripsi/results/transformer_training_results.csv`.


In [10]:
all_results = distilbert_results + [roberta_metrics]
results_df = pd.DataFrame(all_results)

# Urutkan kolom utama
cols_order = [
    'model', 'seed', 'accuracy', 'precision', 'recall', 'f1_binary', 'f1_macro',
    'roc_auc', 'pr_auc', 'training_time_s', 'peak_vram_gb', 'latency_ms', 'throughput_sps'
]
results_df = results_df[[c for c in cols_order if c in results_df.columns]]

# Simpan ke CSV di Google Drive
results_df.to_csv(TRANSFORMER_RESULTS_CSV, index=False)
print(f"✅ Hasil training transformer berhasil disimpan: {TRANSFORMER_RESULTS_CSV}")

print("\n=== TABEL PERBANDINGAN MODEL TRANSFORMER ===")
display(results_df)

print("\n🚀 [SELESAI] File 3: 03_Train_Transformers.ipynb berhasil dieksekusi dengan sempurna!")


✅ Hasil training transformer berhasil disimpan: /content/drive/MyDrive/Skripsi/results/transformer_training_results.csv

=== TABEL PERBANDINGAN MODEL TRANSFORMER ===


,model,seed,accuracy,precision,recall,f1_binary,f1_macro,roc_auc,pr_auc,training_time_s,peak_vram_gb,latency_ms,throughput_sps
0,DistilBERT-base,42,0.923766,0.873551,0.974937,0.921464,0.923701,0.980589,0.966141,3235.636810,2.459541,0.429459,2328.508773
1,DistilBERT-base,123,0.919497,0.863534,0.979261,0.917764,0.919462,0.986117,0.979770,3240.957473,2.460884,0.435690,2295.211350
2,DistilBERT-base,999,0.911720,0.850068,0.980487,0.910632,0.911707,0.976908,0.958374,3241.885302,2.465889,0.421778,2370.914221
3,RoBERTa-base,42,0.972197,0.956878,0.983722,0.970115,0.972062,0.995220,0.991606,6461.200913,5.251205,0.811494,1232.295267



🚀 [SELESAI] File 3: 03_Train_Transformers.ipynb berhasil dieksekusi dengan sempurna!
